# Lab Portal — AI Service trên Kaggle + Cloudflare

Notebook chứa sẵn **mã nguồn, config, contract và regression tests** từ checkout được ghi ở cell snapshot. Không clone `dev`, không dùng `ai-service.zip` cũ. Không chứa model weights hoặc token.

1. Import notebook vào Kaggle; bật **Internet** và **GPU NVIDIA** (runtime dùng một GPU `cuda:0`).
2. Bấm **Add Input** và chọn Dataset **`AI_Service_Dataset`** đã có của bạn. Notebook tự tìm thư mục có `shared-base` và `research-assistant` ở mọi cấp bên dưới `/kaggle/input`; không cần đoán username/slug hay tải lại từ Drive. Có thể dùng ZIP64 còn nguyên vẹn nếu Dataset chưa giải nén.
3. Tạo Kaggle Secret **`AI_INTERNAL_SERVICE_TOKEN`**, bật quyền truy cập cho notebook. Spring dùng cùng giá trị.
4. Chạy từ trên xuống. Chỉ cấu hình dự án sau `PUBLIC READY: PASS` và smoke test. **Run All giữ service/tunnel chạy**; cell dừng mặc định tắt.

URL `trycloudflare.com` thay đổi khi tạo tunnel mới và hết dùng được khi session dừng. Đây là môi trường test. Python nhận token nội bộ; trình duyệt đăng nhập gọi Spring, không gọi Python trực tiếp.

Kế hoạch kiểm thử đầy đủ: `docs/testing/AI_SERVICE_KAGGLE_TEST_PLAN.md` trong dự án. Notebook smoke dùng fixture kỹ thuật ghi rõ, không phải dữ liệu nghiệp vụ thật.


In [ ]:
import base64, hashlib, io, json, os, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

def exactly_one(paths, label):
    paths = sorted(set(Path(p).resolve() for p in paths))
    if len(paths) != 1:
        raise RuntimeError(f"{label}: expected one input, found {paths}. Set ARTIFACT_INPUT explicitly.")
    return paths[0]

def discover_artifacts(input_root):
    directories = [p.parent for p in input_root.rglob("shared-base")
                   if p.is_dir() and (p.parent / "research-assistant").is_dir()]
    archives = [p for p in input_root.rglob("runtime-artifacts*.zip") if p.is_file()]
    return exactly_one(directories or archives, "Runtime artifacts")

def extract_zip(archive, destination):
    if not zipfile.is_zipfile(archive):
        raise zipfile.BadZipFile(f"Invalid/incomplete ZIP: {archive}")
    root = Path(destination).resolve()
    if root.exists():
        raise FileExistsError(root)
    with zipfile.ZipFile(archive) as bundle:
        entries = []
        for member in bundle.infolist():
            name = member.filename.replace("\\", "/")
            target = (root / name).resolve()
            if (not target.is_relative_to(root) or ":" in name.split("/")[0]
                    or ((member.external_attr >> 16) & 0o170000) == 0o120000):
                raise ValueError("Unsafe ZIP entry: " + member.filename)
            entries.append((member, name, target))
        root.mkdir(parents=True)
        for member, name, target in entries:
            if name.endswith("/"):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(member) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
    return root

def require_ready(body):
    fields = ("ready", "profileLoaded", "artifactValidated", "modelLoaded", "adapterLoaded")
    if body.get("status") != "READY" or any(body.get(field) is not True for field in fields):
        raise RuntimeError("Model/adapter is not fully READY: " + json.dumps(body))

def call_json(url, token, *, payload=None, request_id=None, authenticated=True, timeout=110):
    headers = {"X-Request-Id": request_id or "kaggle-" + str(time.time_ns())}
    if authenticated:
        headers["X-Internal-Service-Token"] = token
    data = None
    if payload is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(payload).encode("utf-8")
    try:
        with urlopen(Request(url, headers=headers, data=data), timeout=timeout) as response:
            return response.status, json.load(response)
    except HTTPError as error:
        # Preserve expected 4xx/5xx status and the contract envelope, without echoing headers.
        return error.code, json.loads(error.read().decode("utf-8"))


## 1. Cấu hình input và kiểm tra GPU

Sau khi **Add Input → AI_Service_Dataset**, giữ `ARTIFACT_INPUT = ""` để tự tìm bộ artifacts đã gắn. Nếu có nhiều bộ, nhập đường dẫn chính xác dưới `/kaggle/input`. Thư mục Dataset được đọc trực tiếp. Các ZIP `runtime-artifacts.zip`, `runtime-artifacts-colab.zip` và `runtime-artifacts-kaggle.zip` cũng được nhận diện; ZIP cần đủ dung lượng trống để giải nén. Metadata “30 files / 6 GB” chưa chứng minh đúng model: cell checksum bên dưới vẫn phải PASS đủ 19 file.


In [ ]:
WORK = Path("/kaggle/working/lab-portal-ai")
INPUT = Path("/kaggle/input")
ARTIFACT_INPUT = ""  # Directory with shared-base/ and research-assistant/, or a ZIP64.
PORT = 8090
PYTHON_REQUEST_TIMEOUT = 90   # Test-only setting; Cloudflare edge can time out independently.
GATEWAY_ATTEMPT_TIMEOUT = "100s"
if not INPUT.is_dir():
    raise RuntimeError("Run this notebook inside Kaggle with the artifact Dataset attached.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)
WORK.mkdir(parents=True, exist_ok=True)
print("Attached inputs:", [p.name for p in INPUT.iterdir()])


## 2. Snapshot mã nguồn đã đóng gói

Cell tiếp theo là dữ liệu ZIP base64, không cần chỉnh sửa. SHA-256 kiểm tra bytes của toàn bộ gói và từng file. Sau này cập nhật source, chạy `python scripts/prepare-ai-service-kaggle.py` tại máy phát triển rồi import lại notebook mới.


In [ ]:
# Generated by scripts/prepare-ai-service-kaggle.py; contains source/config/tests, never model weights.
SOURCE_GIT_REVISION = "be045a82c44c2eee33b6f016eba8c2af1912169d"
SOURCE_ZIP_SHA256 = "d379a940e366681cf9da332978331481418f10ba6de4ff078c2cc40624184698"
SOURCE_ZIP_B64 = (
    "..."
    "APcXAAD0SQIAAAA="
)
print("Embedded source:", SOURCE_GIT_REVISION[:12], "SHA256:", SOURCE_ZIP_SHA256)


In [ ]:
archive_bytes = base64.b64decode(SOURCE_ZIP_B64, validate=True)
if hashlib.sha256(archive_bytes).hexdigest() != SOURCE_ZIP_SHA256:
    raise RuntimeError("Embedded source ZIP checksum mismatch.")
SOURCE_ROOT = extract_zip(io.BytesIO(archive_bytes), WORK / ("source-" + str(time.time_ns())))
SOURCE_MANIFEST = json.loads((SOURCE_ROOT / "kaggle-source-manifest.json").read_text())
for name, expected in SOURCE_MANIFEST["files"].items():
    if hashlib.sha256((SOURCE_ROOT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Source file checksum mismatch: " + name)
SERVICE = SOURCE_ROOT / "ai-service"
CONTRACT = json.loads((SOURCE_ROOT / "contracts/ai-runtime-contract.json").read_text())
print("SOURCE CHECK: PASS", len(SOURCE_MANIFEST["files"]), "files; Git", SOURCE_GIT_REVISION[:12])


## 3. Tìm và xác minh toàn bộ model/adapter

Manifest runtime yêu cầu base Qwen và Research adapter; Admin/Lab chạy base model. Không sửa checksum, bỏ qua file, hoặc đánh dấu READY thủ công. Bước này đọc toàn bộ weights để kiểm SHA-256 nên có thể mất vài phút.


In [ ]:
if ARTIFACT_INPUT:
    artifact = Path(ARTIFACT_INPUT)
else:
    artifact = discover_artifacts(INPUT)
if artifact.is_file():
    with zipfile.ZipFile(artifact) as bundle:
        uncompressed_bytes = sum(member.file_size for member in bundle.infolist())
    if shutil.disk_usage(WORK).free < uncompressed_bytes + 2 * 1024**3:
        raise RuntimeError("Not enough working disk for artifact extraction. Attach an extracted Dataset instead.")
    extracted = extract_zip(artifact, WORK / ("weights-" + str(time.time_ns())))
    artifact = exactly_one([p.parent for p in extracted.rglob("shared-base")
                            if (p.parent / "research-assistant").is_dir()], "Extracted artifacts")
ARTIFACT_ROOT = artifact.resolve(strict=True)
subprocess.run([sys.executable, str(SERVICE / "scripts/prepare_runtime_artifacts.py"),
                "--manifest", str(SERVICE / "config/model-artifacts.runtime-t4.json"),
                "verify", "--artifact-root", str(ARTIFACT_ROOT)], check=True)
print("ARTIFACT CHECK: PASS", ARTIFACT_ROOT)
# Existing tests use the service's default artifact path. Link to the verified,
# read-only Dataset rather than duplicating weights or weakening descriptors.
artifact_link = SERVICE / "artifacts"
if artifact_link.exists() or artifact_link.is_symlink():
    if artifact_link.resolve() != ARTIFACT_ROOT:
        raise RuntimeError("Existing test artifact link points at a different Dataset.")
else:
    artifact_link.symlink_to(ARTIFACT_ROOT, target_is_directory=True)


## 4. Môi trường Python 3.12 riêng

Dùng pins trong `requirements-runtime-t4.txt`; không thay Torch/torchvision của kernel Kaggle và không cần restart kernel. Nếu không có wheel phù hợp hoặc dependency xung đột, cell dừng với lỗi gốc. Không tự nâng/hạ version.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("uv executable not found after installation.")
VENV = WORK / "runtime-python312"
if not (VENV / "bin/python").is_file():
    subprocess.run([UV, "venv", "--python", "3.12", str(VENV)], check=True)
PYTHON = str(VENV / "bin/python")
subprocess.run([UV, "pip", "install", "--python", PYTHON, "--index-strategy", "unsafe-best-match",
                "-r", str(SERVICE / "requirements-runtime-t4.txt"), "-r", str(SERVICE / "requirements.txt")],
               cwd=SERVICE, check=True)
subprocess.run([UV, "pip", "check", "--python", PYTHON], check=True)
subprocess.run([PYTHON, "-c", "import torch,transformers,peft,bitsandbytes; "
    "print('Torch',torch.__version__,'Transformers',transformers.__version__,'PEFT',peft.__version__); "
    "assert torch.cuda.is_available(), 'CUDA is unavailable: enable a compatible GPU'; "
    "print('GPU',torch.cuda.get_device_name(0))"], check=True)


## 5. Regression không cần load model

Kiểm tra contract/security/planner và ba assistant bằng hạ tầng test hiện có. Đây là unit/contract evidence; bước này không thay thế inference thật hoặc test quyền qua Spring.


In [ ]:
test_env = os.environ.copy()
test_env.pop("AI_RUNTIME_LOAD_ENABLED", None)
test_env["AI_RUNTIME_LOAD_ENABLED"] = "false"
subprocess.run([PYTHON, "-m", "pytest", "tests", "-q", "--durations=10", "--junitxml=" + str(WORK / "contract-regression.xml")],
               cwd=SERVICE, env=test_env, check=True)
print("LOCAL CONTRACT/REGRESSION: PASS (model-backed inference is checked below)")


## 6. Đọc Secret và khởi động service

Token không được in hoặc ghi ra file kết quả. Chỉ một Uvicorn worker, một GPU. Cell không giết process không thuộc notebook; nếu port đang bận, dùng cell dừng ở cuối rồi chạy lại.


In [ ]:
if "ai_process" in globals() and ai_process.poll() is None:
    raise RuntimeError("Notebook service is already running. Stop it explicitly before restarting.")
import socket
with socket.socket() as port_probe:
    try:
        port_probe.bind(("127.0.0.1", PORT))
    except OSError as error:
        raise RuntimeError("Port is already in use. Stop its owner or choose another PORT.") from error
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("AI_INTERNAL_SERVICE_TOKEN")
if not TOKEN or len(TOKEN) > 1024 or any(not 0x21 <= ord(c) <= 0x7E for c in TOKEN):
    raise ValueError("Secret must be 1-1024 printable ASCII characters without whitespace.")
service_env = os.environ.copy()
service_env.update({
    "AI_INTERNAL_SERVICE_TOKEN": TOKEN, "AI_ENVIRONMENT": "local",
    "AI_MODEL_ARTIFACTS_PATH": str(SERVICE / "config/model-artifacts.runtime-t4.json"),
    "AI_MODEL_ARTIFACT_ROOT": str(ARTIFACT_ROOT),
    "AI_ASSISTANT_PROFILES_PATH": str(SERVICE / "config/assistant-profiles.json"),
    "AI_OUTPUT_SCHEMAS_PATH": str(SERVICE / "config/schemas/structured-output-schemas.json"),
    "AI_RUNTIME_LOAD_ENABLED": "true", "AI_RUNTIME_DEVICE": "cuda:0",
    "AI_REQUEST_TIMEOUT_SECONDS": str(PYTHON_REQUEST_TIMEOUT),
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "CUDA_VISIBLE_DEVICES": "0",
    "TOKENIZERS_PARALLELISM": "false",
})
LOG = WORK / "ai-service.log"
with LOG.open("w") as log:
    ai_process = subprocess.Popen([PYTHON, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1",
        "--port", str(PORT), "--workers", "1", "--no-access-log"],
        cwd=SERVICE, env=service_env, stdout=log, stderr=subprocess.STDOUT)
print("Service PID:", ai_process.pid)


In [ ]:
LOCAL_URL = f"http://127.0.0.1:{PORT}"
READY = False
deadline = time.monotonic() + 600
while time.monotonic() < deadline:
    if ai_process.poll() is not None:
        raise RuntimeError("AI process exited. Enable SHOW_LOGS in the debug cell.")
    try:
        status, readiness = call_json(LOCAL_URL + "/ready", TOKEN, timeout=10)
        if status == 200:
            require_ready(readiness)
            READY = True
            break
    except (URLError, TimeoutError, OSError, ValueError, RuntimeError):
        pass
    time.sleep(5)
if not READY:
    raise TimeoutError("Model not READY after 10 minutes; no tunnel has been opened.")
status, MODEL_INFO = call_json(LOCAL_URL + "/model-info", TOKEN)
if status != 200:
    raise RuntimeError("model-info failed")
require_ready(MODEL_INFO)
expected_identity = json.loads((SERVICE / "config/model-artifacts.runtime-t4.json").read_text())["baseModel"]["artifact"]["identity"]
if MODEL_INFO.get("artifactIdentity") != expected_identity:
    raise RuntimeError("Loaded base model identity differs from packaged runtime descriptor.")
print("LOCAL READY: PASS")
print(json.dumps(MODEL_INFO, indent=2, ensure_ascii=False))


## 7. Tạo Cloudflare Quick Tunnel và kiểm tra từ URL công khai

Tải `cloudflared` từ release chính thức. Dùng HTTP/2 cho kết nối ra ngoài. Mọi route vẫn yêu cầu token nội bộ; URL ngẫu nhiên không thay thế authentication.


In [ ]:
if not READY or ai_process.poll() is not None:
    raise RuntimeError("The local AI service must be READY first.")
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    raise RuntimeError("A notebook tunnel is already running; stop it explicitly first.")
CLOUDFLARED = WORK / "cloudflared"
if not CLOUDFLARED.is_file():
    download = WORK / "cloudflared.download"
    with urlopen("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", timeout=60) as response, download.open("wb") as output:
        shutil.copyfileobj(response, output)
    download.chmod(0o700)
    download.replace(CLOUDFLARED)
subprocess.run([str(CLOUDFLARED), "--version"], check=True)
TUNNEL_LOG = WORK / "cloudflared.log"
with TUNNEL_LOG.open("w") as log:
    tunnel_process = subprocess.Popen([str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--protocol", "http2",
        "--url", LOCAL_URL], stdout=log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for attempt in range(60):
    if tunnel_process.poll() is not None:
        raise RuntimeError("Cloudflare tunnel exited. Enable SHOW_LOGS in the debug cell.")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text(errors="replace"))
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(2)
if not PUBLIC_URL:
    raise TimeoutError("No Cloudflare URL after 2 minutes.")
for attempt in range(18):
    try:
        status, public_ready = call_json(PUBLIC_URL + "/ready", TOKEN, timeout=10)
        if status == 200:
            require_ready(public_ready)
            break
    except (URLError, TimeoutError, OSError, ValueError, RuntimeError):
        pass
    time.sleep(5)
else:
    raise RuntimeError("Public readiness failed. Do not configure Spring yet.")
print("PUBLIC READY: PASS", PUBLIC_URL)


## 8. Smoke qua tunnel: authentication, contract và inference

Read snapshot hợp lệ có thể dùng 0 token. Planner và draft phải thực sự chạy model; kiểm riêng proposal, Admin account draft và report với đúng payload Spring. Nếu smoke fail, chưa đánh dấu runtime/E2E PASS.


In [ ]:
# Load only the existing pure fixture functions, without importing app.main or
# the test module (those imports would initialize another service/model).
fixture_code = """
import ast, json
from pathlib import Path
def load(name):
    path = Path('tests') / (name + '.py')
    tree = ast.parse(path.read_text(encoding='utf-8'))
    function = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == '_request')
    namespace = {}; exec(compile(ast.Module(body=[function], type_ignores=[]), str(path), 'exec'), namespace)
    return namespace['_request']
admin = load('test_admin_mvp')('admin.system.summary','SYSTEM',None)
lab = load('test_lab_mvp')('lab.policy.read','LABORATORY',10)
research = load('test_research_mvp')('research.group.summary','GROUP',30)
print(json.dumps([admin,lab,research]))
"""
fixtures = subprocess.run([PYTHON, "-c", fixture_code], cwd=SERVICE, env=test_env,
                          check=True, capture_output=True, text=True)
smoke_payloads = json.loads(fixtures.stdout)
smoke_results = []
for use_token, authenticated in [("", False), ("invalid-kaggle-smoke-token", True)]:
    status, body = call_json(PUBLIC_URL + "/health", use_token, authenticated=authenticated)
    if status != 401 or body.get("errorCode") != "AI_INTERNAL_AUTH_FAILED":
        raise RuntimeError("Public authentication guard failed")
smoke_results.append({"case": "missing/wrong internal token", "result": "PASS"})
for payload in smoke_payloads:
    payload["locale"] = "vi"
    started = time.monotonic()
    status, body = call_json(PUBLIC_URL + CONTRACT["routes"]["chat"]["path"], TOKEN, payload=payload)
    if (status != 200 or body.get("assistantKey") != payload["assistantKey"] or not body.get("answer")
            or body.get("metadata", {}).get("grounding") != "AUTHORIZED_SNAPSHOT"
            or body.get("metadata", {}).get("safeRefusal") is True):
        raise RuntimeError("Authorized snapshot smoke failed for " + payload["assistantKey"] + ": " + json.dumps(body))
    result = {"case": payload["assistantKey"], "result": "PASS", "seconds": round(time.monotonic()-started, 2),
              "promptTokens": body["promptTokens"], "completionTokens": body["completionTokens"],
              "grounding": body["metadata"]["grounding"]}
    smoke_results.append(result)
    print(json.dumps(result), "\nAnswer for manual review:", body["answer"])
# Separate actual generation from deterministic reads: planner plus structured proposal.
proposal = next(p for p in smoke_payloads if p["assistantKey"] == "RESEARCH_ASSISTANT")
proposal["input"] = "Return a concise task proposal JSON only for group30/project20, advisory draft."
authorized = proposal["authorizedContext"]
authorized["context"]["draftOnly"] = True
authorized["resources"].append({"resourceType": "PROJECT", "resourceId": 20})
authorized["allowedTools"][0].update(toolId="research.task.proposal.draft", parentResourceType="PROJECT",
    argumentNames=["resource", "parentResource"], riskBoundary="DRAFT_ONLY")
status, draft = call_json(PUBLIC_URL + CONTRACT["routes"]["chat"]["path"], TOKEN, payload=proposal)
if (status != 200 or draft.get("metadata", {}).get("safeRefusal") is True
        or draft.get("promptTokens", 0) < 1 or draft.get("completionTokens", 0) < 1):
    raise RuntimeError("Actual Research generation failed: " + json.dumps(draft))
parsed_draft = json.loads(draft["answer"])
if parsed_draft.get("groupRef") != 30 or parsed_draft.get("projectRef") != 20:
    raise RuntimeError("Generated proposal references differ from authorized resources")
smoke_results.append({"case": "Research actual structured generation", "result": "PASS"})
# Exercise Admin draft and the exact real-Spring report context shape before handoff.
admin_draft = json.loads(json.dumps(smoke_payloads[0]))
admin_draft["input"] = "Draft review actions for authorized account9; do not lock or update it."
admin_context = admin_draft["authorizedContext"]
admin_context["context"].update(targetUser={"id":9,"status":"ACTIVE","active":True}, draftOnly=True)
admin_context["resources"] = [{"resourceType":"USER_ACCOUNT","resourceId":9}]
admin_context["allowedTools"][0].update(toolId="admin.account.action.draft", resourceType="USER_ACCOUNT", riskBoundary="DRAFT_ONLY")
status, account = call_json(PUBLIC_URL + CONTRACT["routes"]["chat"]["path"], TOKEN, payload=admin_draft)
if status != 200 or account.get("metadata", {}).get("safeRefusal") is True:
    raise RuntimeError("Admin draft generation failed: " + json.dumps(account))
parsed_account = json.loads(account["answer"])
if parsed_account.get("subject") != "9" or parsed_account.get("requiresHumanReview") is not True:
    raise RuntimeError("Admin draft identity/advisory contract failed")
smoke_results.append({"case":"Admin actual account draft", "result":"PASS"})
report_payload = json.loads((SERVICE / "tests/fixtures/report-review-spring.json").read_text(encoding="utf-8"))
status, report = call_json(PUBLIC_URL + CONTRACT["routes"]["chat"]["path"], TOKEN, payload=report_payload)
if status != 200 or report.get("metadata", {}).get("safeRefusal") is True:
    raise RuntimeError("Real-Spring report fixture generation failed: " + json.dumps(report))
parsed_report = json.loads(report["answer"])
if (parsed_report.get("reportRef") != report_payload["authorizedContext"]["resources"][0]["resourceId"]
        or parsed_report.get("advisoryOnly") is not True or parsed_report.get("requiresHumanReview") is not True):
    raise RuntimeError("Report draft identity/advisory contract failed")
smoke_results.append({"case":"Research real-Spring report fixture generation", "result":"PASS"})
status, planner = call_json(PUBLIC_URL + CONTRACT["routes"]["toolRequest"]["path"], TOKEN,
                          payload=CONTRACT["toolPlanningRequest"]["example"])
if (status != 200 or planner.get("decision") != "TOOL_REQUEST"
        or (planner.get("toolRequest") or {}).get("toolId") != "lab.slot.read"):
    raise RuntimeError("Planner smoke failed: " + json.dumps(planner))
smoke_results.append({"case": "planner canonical tool selection", "result": "PASS"})
# Replay the two research requests that were incorrectly routed as Lab shift creation.
for prompt, tool_id, resource, parent, read_tool in [
    ("Soạn một đề xuất task cho nhóm 19 dự án 16, chỉ draft, không tạo record.",
     "research.task.proposal.draft", {"resourceType":"GROUP","resourceId":19},
     {"resourceType":"PROJECT","resourceId":16}, "research.group.summary"),
    ("Gợi ý cách làm task 88, chỉ draft tư vấn, không đổi trạng thái task.",
     "research.task.suggestion.draft", {"resourceType":"TASK","resourceId":88},
     None, "research.assigned.task.read"),
]:
    candidates = [{"assistantKey":"RESEARCH_ASSISTANT", "schemaVersion":"v1", "toolId":tid,
                   "description":"Authorized " + tid, "resource":resource,
                   "parentResource":parent if tid == tool_id else None} for tid in (read_tool, tool_id)]
    status, routed = call_json(PUBLIC_URL + CONTRACT["routes"]["toolRequest"]["path"], TOKEN,
                               payload={"input":prompt,"locale":"vi","candidates":candidates})
    if (status != 200 or routed.get("decision") != "TOOL_REQUEST"
            or (routed.get("toolRequest") or {}).get("toolId") != tool_id):
        raise RuntimeError("Research planner regression: " + json.dumps(routed))
    smoke_results.append({"case":"Research planner " + tool_id, "result":"PASS"})
# Exact Spring conversation envelope plus project/group/proposal candidates.
spring_planner_payload = json.loads((SERVICE / "tests/fixtures/planner-research-context.json").read_text(encoding="utf-8"))
status, spring_plan = call_json(PUBLIC_URL + CONTRACT["routes"]["toolRequest"]["path"], TOKEN, payload=spring_planner_payload)
if (status != 200 or spring_plan.get("decision") != "TOOL_REQUEST"
        or (spring_plan.get("toolRequest") or {}).get("toolId") != "research.task.proposal.draft"):
    raise RuntimeError("Spring conversation planner regression: " + json.dumps(spring_plan))
smoke_results.append({"case":"Spring conversation research planner", "result":"PASS"})
(WORK / "smoke-results.json").write_text(json.dumps({"gitRevision": SOURCE_GIT_REVISION,
    "sourceZipSha256": SOURCE_ZIP_SHA256, "publicUrl": PUBLIC_URL, "results": smoke_results}, indent=2))
print("PUBLIC SMOKE: PASS. Next: connect Spring and test authenticated users with real fixtures.")


In [ ]:
# Runtime assertions use real model HTTP; synthetic contexts do not prove Spring authorization.
verify_env = os.environ.copy()
verify_env["AI_INTERNAL_SERVICE_TOKEN"] = TOKEN
VERIFY_SCRIPT = str(SERVICE / "scripts/verify_runtime.py")
def verify_runtime_group(label, selectors="all", concurrency=4, repeat=1, verify_artifacts=False):
    destination = WORK / (label + "-" + str(time.time_ns()))
    command = [PYTHON, VERIFY_SCRIPT, "--base-url", LOCAL_URL, "--source-root", str(SOURCE_ROOT),
               "--expected-source-sha", SOURCE_ZIP_SHA256, "--case-ids", selectors,
               "--concurrency", str(concurrency), "--repeat", str(repeat), "--output-dir", str(destination)]
    if verify_artifacts:
        command.extend(["--artifact-root", str(ARTIFACT_ROOT)])
    result = subprocess.run(command, cwd=SERVICE, env=verify_env)
    evidence = json.loads((destination / "runtime-results.json").read_text(encoding="utf-8"))
    print("RUNTIME EVIDENCE", str(destination), "source", evidence.get("sourceZipSha256"),
          "passed", evidence["passed"], "failed", evidence["failed"], "blocked", evidence.get("blocked"))
    for row in evidence["results"]:
        if not row["passed"]:
            print("FAILED VARIANT", row["id"], json.dumps(row, ensure_ascii=False))
    if result.returncode:
        raise RuntimeError("Runtime verification did not pass; preserved evidence: " + str(destination))
    return evidence
runtime_report = verify_runtime_group("runtime-corpus", verify_artifacts=True)
print("KAGGLE RUNTIME CORPUS: PASS", len(runtime_report["results"]), "variants")


In [ ]:
# Warm functional workload, not a production capacity/SLA benchmark.
load_ids = ["admin-account-vi", "lab-booking-vi", "research-proposal-vi", "research-report-vi"]
load_reports = []
for count in (1, 2, 4):
    load_reports.append(verify_runtime_group("runtime-load-" + str(count), ",".join(load_ids[:count]), concurrency=count))
print("KAGGLE 1/2/4 GENERATION: PASS")


In [ ]:
mixed_report = verify_runtime_group("runtime-mixed", ",".join([
    "admin-account-vi", "lab-booking-vi", "research-report-vi", "planner-vi-envelope"]), concurrency=4)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True, check=True)
print("GPU AFTER TEST (not peak VRAM):", gpu.stdout.strip())
print("KAGGLE MIXED PLANNER/GENERATION: PASS")


## 9. Nối ngay vào dự án local

Cell tạo **hai file cấu hình không chứa token** trong `/kaggle/working/lab-portal-ai`:

* `connect-lab-portal.ps1`: tải về máy Windows, **dot-source** trong terminal sẽ khởi động Spring: `. .\connect-lab-portal.ps1`. Script hỏi token bằng input ẩn, kiểm `/ready`, rồi đặt ba biến môi trường cho đúng terminal. Đóng/restart Spring hoặc chạy lại backend từ terminal này với profile `local` và các biến DB/Redis/JWT/Face hiện có. IDE đang chạy cần Run Configuration có cùng ba biến rồi restart.
* `docker-compose.ai-kaggle.yml`: khi dùng Docker Compose local, tải file này vào root repo; đặt token bằng script trên, rồi `docker compose -f docker-compose.yml -f docker-compose.ai-kaggle.yml up -d --no-deps backend`. Container backend phải đã build và DB/Redis/Face đã chạy. Override cần thiết vì compose local hiện hardcode URL AI nội bộ.

Không đổi frontend API URL sang Kaggle. Trình duyệt tiếp tục gọi `/api/ai/chat` trên Spring. Quick Tunnel, GPU session và token phải còn hợp lệ trong suốt lúc test.


In [ ]:
powershell = r"""# Dot-source this file in the terminal that will launch Spring.
$ErrorActionPreference = 'Stop'
$secureAiToken = Read-Host 'AI_INTERNAL_SERVICE_TOKEN (same Kaggle Secret)' -AsSecureString
$tokenPointer = [Runtime.InteropServices.Marshal]::SecureStringToBSTR($secureAiToken)
try {
    $aiToken = [Runtime.InteropServices.Marshal]::PtrToStringBSTR($tokenPointer)
} finally {
    [Runtime.InteropServices.Marshal]::ZeroFreeBSTR($tokenPointer)
}
try {
    $aiBaseUrl = '__PUBLIC_URL__'
    $aiReady = Invoke-RestMethod -Uri ($aiBaseUrl + '/ready') -Headers @{
        'X-Internal-Service-Token' = $aiToken
        'X-Request-Id' = ('windows-ready-' + [guid]::NewGuid().ToString('N'))
    } -TimeoutSec 20
    foreach ($field in @('ready','profileLoaded','artifactValidated','modelLoaded','adapterLoaded')) {
        if ($aiReady.$field -ne $true) { throw ('AI is not READY: ' + $field) }
    }
    $env:AI_GATEWAY_BASE_URL = $aiBaseUrl
    $env:AI_GATEWAY_ATTEMPT_TIMEOUT = '__GATEWAY_TIMEOUT__'
    $env:AI_INTERNAL_SERVICE_TOKEN = $aiToken
    Write-Host ('AI READY. Gateway set for this terminal: ' + $aiBaseUrl)
    Write-Host 'Restart/launch Spring with profile local in this terminal; keep existing DB/Redis/JWT/Face settings.'
} finally {
    $aiToken = $null
    $secureAiToken.Dispose()
}
""".replace("__PUBLIC_URL__", PUBLIC_URL).replace("__GATEWAY_TIMEOUT__", GATEWAY_ATTEMPT_TIMEOUT)
(WORK / "connect-lab-portal.ps1").write_text(powershell, encoding="utf-8-sig")
override = f"""services:
  backend:
    environment:
      AI_GATEWAY_BASE_URL: {PUBLIC_URL}
      AI_GATEWAY_ATTEMPT_TIMEOUT: {GATEWAY_ATTEMPT_TIMEOUT}
      AI_INTERNAL_SERVICE_TOKEN: ${{AI_INTERNAL_SERVICE_TOKEN:?Set the same Kaggle Secret in this terminal}}
"""
(WORK / "docker-compose.ai-kaggle.yml").write_text(override)
(WORK / "connection.json").write_text(json.dumps({"AI_GATEWAY_BASE_URL": PUBLIC_URL,
    "AI_GATEWAY_ATTEMPT_TIMEOUT": GATEWAY_ATTEMPT_TIMEOUT, "tokenSource": "Kaggle Secret AI_INTERNAL_SERVICE_TOKEN",
    "gitRevision": SOURCE_GIT_REVISION}, indent=2))
print("AI_GATEWAY_BASE_URL=" + PUBLIC_URL)
print("AI_GATEWAY_ATTEMPT_TIMEOUT=" + GATEWAY_ATTEMPT_TIMEOUT)
print("Download:", WORK / "connect-lab-portal.ps1", "and", WORK / "docker-compose.ai-kaggle.yml")


## 10. Log khi cần và dừng thủ công

Đổi `SHOW_LOGS` thành `True` để xem log. Chỉ đổi `STOP_SERVICES` thành `True` khi đã test xong. Không save/publish notebook có token hoặc dữ liệu người dùng thật trong output.

Nguồn hướng dẫn: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks), [Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/), [uv environments](https://docs.astral.sh/uv/pip/environments/). Cloudflare Quick Tunnel không cung cấp SSE; contract hiện tại dùng JSON request/response.


In [ ]:
SHOW_LOGS = False
if SHOW_LOGS:
    for path in (WORK / "ai-service.log", WORK / "cloudflared.log"):
        if path.exists():
            text = path.read_text(errors="replace")[-12000:]
            if globals().get("TOKEN"):
                text = text.replace(TOKEN, "[REDACTED]")
            print(path.name, "\n", text)


In [ ]:
STOP_SERVICES = False  # Set True and run this cell ONLY when testing is finished.
if STOP_SERVICES:
    for name in ("tunnel_process", "ai_process"):
        process = globals().get(name)
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=15)
    READY = False
    print("Stopped notebook-owned processes.")
else:
    print("Service/tunnel kept running. Set STOP_SERVICES=True here when finished.")
